# 01 · Multi-event: does the loss framework (L_rank, L_mul, MMV) help?

The paper's central claim (§2.3, Eq. 5): ranking the events **within** a patient (`L_mul`)
improves multi-event survival prediction. This notebook tests it on

* **sim_base** — simulator with known true CIFs (oracle row = the ceiling),
* **EBMT** — real bone-marrow-transplant data, 5 non-exclusive events (semi-competing),
* **hsa_synthetic** — the 2-event dataset bundled with the repo,

against Cox PH, DeepHit, DSM and MENSA (as published), with 5 seeds, each on its own
train/val/test split.

Run with `SMOKE_TEST = True` first, then `False`. Results: `/kaggle/working/results/01_multievent_lmul/`
and a zip next to it. Resume after a timeout by attaching this notebook's output as input.

## Options

In [ ]:
NAME = "01_multievent_lmul"

# Datasets: sim_base = simulator base scenario (known true CIFs -> oracle row),
# hsa_synthetic = the 2-event dataset shipped with the repo, ebmt = real EBMT (5 events).
DATASETS = ["sim_base", "ebmt", "hsa_synthetic"]
SEEDS    = [0, 1, 2, 3, 4]      # seed s = init seed s AND train/val/test split s

BASELINES = ["coxph", "deephit_paper", "dsm_paper", "mensa_paper"]
# Our model, loss ablation (paper Eq. 2-5, plus MMV):
OUR_RECIPES = [
    "nllpch",                        # L_PCH
    "nllpch_event_ranking",          # L_PCH + L_mul            (isolates L_mul)
    "nllpch_sample_ranking",         # L_PCH + L_rank
    "nllpch_sample_event_ranking",   # L_PCH + L_rank + L_mul   (the paper's recipe)
    "nllpch_mmv_rank",               # ... + MMV                (full recipe)
]
MMV_LAMBDA_V = 1.0                   # lambda_v for the MMV recipe (scale balancing)

RUN_INFERENCE_DEMO = True            # predict per-event curves for new EBMT patients at the end

# --- execution --------------------------------------------------------------
SMOKE_TEST   = False    # False = the real run. True: 2 epochs, 1 seed, reduced plan, only to check
                        # that everything runs (~10-30 min); smoke results go to a separate folder.
SMOKE_EPOCHS = 2
WORKERS      = None     # parallel runs; None = 2 on Kaggle (4 vCPU), 4 locally
TIME_BUDGET_MIN = 11 * 60   # stop launching new runs after this (Kaggle sessions end at 12 h)
INSTALL_DEPS = True     # Kaggle only
GIT_URL      = "https://github.com/Parsagh05/dmmst.git"   # code is cloned from here (nothing to upload)
CLEAN_UP_AT_END = True  # Kaggle: delete the working repo copy (models, caches) after the report

## Setup (identical in every notebook)

In [ ]:
# ---- 1. get the code --------------------------------------------------------------
import os, sys, json, shutil, subprocess, time, hashlib
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()

def _is_repo(p):
    return (p / "sat" / "finetune.py").is_file() and (p / "conf").is_dir()

if ON_KAGGLE:
    WORK = Path("/kaggle/working")
    REPO = WORK / "dmmst"
    if not _is_repo(REPO):
        # prefer the code dataset over repo copies inside attached notebook outputs (stale code)
        cands = [f.parent.parent for f in Path("/kaggle/input").rglob("finetune.py")
                 if f.parent.name == "sat" and _is_repo(f.parent.parent)]
        cands.sort(key=lambda p: ("code" not in str(p).lower(), len(p.parts)))
        src = cands[0] if cands else None
        if src is not None:
            print("copying code from attached dataset:", src)
            shutil.copytree(src, REPO, ignore=shutil.ignore_patterns(
                ".git", ".venv", "model-hub", "__pycache__", "results", "tb_logs"))
        elif GIT_URL:
            subprocess.run(["git", "clone", "--depth", "1", GIT_URL, str(REPO)], check=True)
        else:
            raise RuntimeError("Attach the dmmst code dataset (notebooks/new/README.md, step 1) or set GIT_URL.")
    RESULTS_ROOT = WORK / "results"
else:
    REPO = Path.cwd().resolve()
    while not _is_repo(REPO) and REPO != REPO.parent:
        REPO = REPO.parent
    RESULTS_ROOT = REPO / "results" / "new"
assert _is_repo(REPO), REPO
os.chdir(REPO)
sys.path.insert(0, str(REPO))
CODE_FINGERPRINT = hashlib.md5(b"".join(
    p.read_bytes() for p in sorted((REPO / "sat").rglob("*.py")) + sorted((REPO / "conf").rglob("*.yaml"))
)).hexdigest()[:10]
print("repo:", REPO, "| code fingerprint:", CODE_FINGERPRINT)

In [ ]:
# ---- 2. dependencies (Kaggle) ------------------------------------------------------
# numpy is NEVER moved: pandas/pyarrow in the image are compiled against it.
if ON_KAGGLE and INSTALL_DEPS:
    import numpy as _np, torch
    _tv = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2])
    pkgs = ["transformers==4.50.0", "datasets==3.4.1", "tokenizers>=0.21,<0.22", "evaluate>=0.4.3",
            "accelerate>=1.4.0", "hydra-core>=1.3.2", "hydra-colorlog>=1.2.0",
            "torchsurv" if _tv >= (2, 8) else "torchsurv==0.1.4",
            "h5py>=3.13", "logdecorator>=2.5", "einops==0.8.1", "torchtuples>=0.2.2",
            "numba", "nvidia-ml-py", "polars", "tqdm", "tabulate",
            "scikit-survival" if int(_np.__version__.split(".")[0]) >= 2 else "scikit-survival==0.22.2",
            f"numpy=={_np.__version__}"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
import torch
print("torch", torch.__version__, "| cuda" if torch.cuda.is_available() else "| cpu")

In [ ]:
# ---- 3. runner ----------------------------------------------------------------------
from IPython.display import Markdown, display
from scripts.runner import Run, Runner, dataset_overrides
from scripts.report import build as build_report

RUN_NAME = NAME + ("_smoke" if SMOKE_TEST else "")
r = Runner(repo=REPO, results_dir=RESULTS_ROOT / RUN_NAME,
           workers=WORKERS or (2 if ON_KAGGLE else 4),
           smoke_epochs=SMOKE_EPOCHS if SMOKE_TEST else None, gpu="0")

# Resume: attach this notebook's previous output ("Add Input" -> your notebook) and the
# finished runs are copied back, so only the missing ones are trained.
if ON_KAGGLE:
    n_copied = 0
    for d in Path("/kaggle/input").rglob(RUN_NAME):
        if d.is_dir() and d != r.results_dir:
            for f in d.glob("*.json"):
                if not (r.results_dir / f.name).exists():
                    shutil.copy2(f, r.results_dir / f.name); n_copied += 1
    print(f"resumed {n_copied} files from attached inputs")
(r.results_dir / "environment.json").write_text(json.dumps({
    "code_fingerprint": CODE_FINGERPRINT, "torch": torch.__version__,
    "cuda": torch.cuda.is_available(), "smoke": SMOKE_TEST, "name": RUN_NAME}, indent=2))
print("results ->", r.results_dir)

## Datasets

In [ ]:
# ---- datasets -------------------------------------------------------------------------
def ensure_dataset(ds):
    """Simulated scenarios and EBMT are generated on the fly; hsa_synthetic ships with the repo."""
    if ds == "hsa_synthetic" or (REPO / "data" / ds / "meta.json").is_file():
        return
    if ds == "ebmt":
        subprocess.run([sys.executable, "scripts/prepare_ebmt.py"], check=True)
    elif ds.startswith("sim_"):
        subprocess.run([sys.executable, "-m", "sat.data.simulate", "--scenario", ds[4:],
                        "--out", f"data/{ds}"], check=True, stdout=subprocess.DEVNULL)
    else:
        raise ValueError(ds)

def group(ds):
    return "hsa_synthetic" if ds == "hsa_synthetic" else "multievent"

def ds_overrides(ds):
    # hsa keeps its own configs; score every event at its own horizons like the others
    return ["per_event_horizons=true"] if ds == "hsa_synthetic" else dataset_overrides(ds, REPO)

def has_truth(ds):
    return (REPO / "data" / ds / "truth.npz").is_file()

In [ ]:
if SMOKE_TEST:
    SEEDS = SEEDS[:1]
for ds in DATASETS:
    ensure_dataset(ds)
    r.prepare(f"{group(ds)}/survival", ds_overrides(ds), label=ds)
    meta = REPO / "data" / ds / "meta.json"
    if meta.is_file():
        m = json.loads(meta.read_text())
        print(ds, {k: m.get(k) for k in ["n", "num_events", "event_rates", "frac_subjects_multiple_events", "unobserved_fraction"] if k in m})

## Plan and run
Seed-major order, so if the session ends early every finished seed is a complete comparison.

In [ ]:
runs = []
for s in SEEDS:
    so = [f"seed={s}", f"split_seed={s}"]
    for ds in DATASETS:
        g, dov, base = group(ds), ds_overrides(ds), {"dataset": ds, "seed": s}
        if has_truth(ds):
            runs.append(Run(f"{ds}__oracle__s{s}", "oracle", f"{g}/survival", dov + so, ds, base | {"model": "oracle"}))
        for m in BASELINES:
            runs.append(Run(f"{ds}__{m}__s{s}", "coxph" if m == "coxph" else "finetune",
                            f"{g}/{m}", dov + so, ds, base | {"model": m}))
        for rec in OUR_RECIPES:
            ov, info = [f"tasks/losses={rec}"], base | {"model": "ours", "recipe": rec}
            if "mmv" in rec:
                ov.append(f"mmv_variance_weight={MMV_LAMBDA_V}")
                info["lambda_v"] = MMV_LAMBDA_V
            runs.append(Run(f"{ds}__ours_{rec}__s{s}", "finetune", f"{g}/survival", dov + so + ov, ds, info))
print(len(runs), "runs")
r.run_many(runs, time_budget_min=TIME_BUDGET_MIN)

## Results
* **C_td**: across-patient ranking per event (SurvTRACE protocol, per-event horizons).
* **Within-subject C** (`within_ctd`): the paper's Eq. 5 pairs — is the event that happened first ranked more likely at that time? `within_ctd_km` is the population-order reference; `within_ctd_gain` = what the model learned beyond it.
* The **paired** table compares each recipe with L_PCH on the same seed/split.

In [ ]:
# ---- report + zip -----------------------------------------------------------------------
df = build_report([r.results_dir], r.results_dir / "report")
rep = r.results_dir / "report" / "REPORT.md"
if rep.is_file():
    display(Markdown(rep.read_text(encoding="utf-8")))
print("zip:", r.zip())

## Predict for new patients (EBMT)
Per-event survival curves for patients the model never saw, from the paper's recipe (seed 0). Replace `NEW_PATIENTS` with real rows.

In [ ]:
if RUN_INFERENCE_DEMO and "ebmt" in DATASETS:
    import pandas as pd
    from sat.data.dataset.parse_multievent import _clean
    rec = "nllpch_sample_event_ranking" if "nllpch_sample_event_ranking" in OUR_RECIPES else OUR_RECIPES[0]
    tag = f"ebmt__ours_{rec}__s{SEEDS[0]}"
    NEW_PATIENTS = pd.DataFrame([
        {"id": 90001, "year": "1995-1998", "agecl": "20-40", "proph": "no",  "match": "no gender mismatch"},
        {"id": 90002, "year": "1985-1989", "agecl": ">40",   "proph": "yes", "match": "gender mismatch"},
        {"id": 90003, "year": "1990-1994", "agecl": "<=20",  "proph": "no",  "match": "no gender mismatch"},
    ])
    cols = ["year", "agecl", "proph", "match"]
    recs = [{"id": int(p.id), "x": " ".join(f"c_{c}_{_clean(p[c])}" for c in cols),
             "modality": [0] * len(cols), "numerics": [1.0] * len(cols)} for _, p in NEW_PATIENTS.iterrows()]
    infer_json = REPO / "data" / "ebmt" / "new_patients.json"
    pd.DataFrame(recs).to_json(infer_json, orient="records", lines=True)
    ok, _, _ = r.sat("infer", "multievent/survival",
                     ds_overrides("ebmt") + [f"modelname={tag}", f"tasks/losses={rec}", f"infer_data={infer_json}"],
                     label="infer_ebmt")
    names = json.loads((REPO / "data" / "ebmt" / "meta.json").read_text())["event_names"]
    out = REPO / "data" / "model-hub" / "ebmt" / tag
    for k, nm in enumerate(names):
        f = out / f"inference_survival{k}.csv"
        if f.is_file():
            print(f"\n{nm}: predicted P(no event yet) at the duration cuts")
            display(pd.read_csv(f).set_index("id").round(3))

## Clean up
Keeps the Kaggle output small (results + zip only) and stops a later session from picking up this repo copy instead of the code dataset.

In [ ]:
if ON_KAGGLE and CLEAN_UP_AT_END:
    shutil.rmtree(REPO, ignore_errors=True)
    print("removed", REPO, "- results are in", r.results_dir)